# 00b – Linear Algebra Review for Quantum Computing and QAOA

This notebook provides a pure NumPy linear algebra review covering statevectors, inner/outer products, unitary and Hermitian operators, Kronecker products, and the mathematical construction of the QAOA Max-Cut cost Hamiltonian.

For every section:
1. We provide the formal mathematical foundation and operator algebra.
2. We include an analytical **Prediction Check** deriving exact numerical expectations before execution.
3. We execute concise, well-commented NumPy code using `@`, `.conj().T`, `np.kron`, and `np.allclose` with strict verification assertions.

## 1. Statevectors & Normalization

**Mathematical Definition:**
A single-qubit quantum state is represented as a unit vector in a 2-dimensional complex Hilbert space $\mathbb{C}^2$:
$$|\psi\rangle = \alpha |0\rangle + \beta |1\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}, \quad \alpha, \beta \in \mathbb{C}$$
Under the Born interpretation of quantum mechanics, total measurement probability across all basis outcomes must sum to unity, requiring unit Euclidean norm ($L_2$ norm):
$$\langle \psi | \psi \rangle = \|\psi\|_2^2 = |\alpha|^2 + |\beta|^2 = 1.0$$
The canonical computational basis states and equatorial superposition states are:
$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$
$$|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix}, \quad |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -1 \end{pmatrix}$$

**Prediction Check:**
- $\||0\rangle\|_2 = \sqrt{1^2 + 0^2} = 1.0$
- $\||1\rangle\|_2 = \sqrt{0^2 + 1^2} = 1.0$
- $\||+\rangle\|_2 = \sqrt{(1/\sqrt{2})^2 + (1/\sqrt{2})^2} = \sqrt{0.5 + 0.5} = 1.0$
- $\||-\rangle\|_2 = \sqrt{(1/\sqrt{2})^2 + (-1/\sqrt{2})^2} = \sqrt{0.5 + 0.5} = 1.0$

In [1]:
# Experiment 1: Define basis states and verify normalization
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Define computational basis states |0> and |1>
ket0 = np.array([1.0, 0.0], dtype=complex)
ket1 = np.array([0.0, 1.0], dtype=complex)

# Define superposition basis states |+> and |->
ket_plus = (ket0 + ket1) / np.sqrt(2)
ket_minus = (ket0 - ket1) / np.sqrt(2)

states = {
    "|0>": ket0,
    "|1>": ket1,
    "|+>": ket_plus,
    "|->": ket_minus
}

print("=== Statevector Verification & Norms ===")
for name, state in states.items():
    norm = np.linalg.norm(state)
    print(f"{name:5s} = {state} | Norm = {norm:.6f}")
    assert np.isclose(norm, 1.0), f"{name} must have unit norm!"

print("\nNormalization Check: ALL PASS (All statevectors have unit norm 1.0)")

=== Statevector Verification & Norms ===
|0>   = [1.+0.j 0.+0.j] | Norm = 1.000000
|1>   = [0.+0.j 1.+0.j] | Norm = 1.000000
|+>   = [0.70710678+0.j 0.70710678+0.j] | Norm = 1.000000
|->   = [ 0.70710678+0.j -0.70710678+0.j] | Norm = 1.000000

Normalization Check: ALL PASS (All statevectors have unit norm 1.0)


## 2. Inner & Outer Products: Orthogonality and Completeness

**Mathematical Definition:**
Using Dirac bra-ket notation:
1. **Inner Product (Bracket $\langle \phi | \psi \rangle$):**
   Computed via conjugate transpose matrix multiplication:
   $$\langle \phi | \psi \rangle = \phi^\dagger \psi = \sum_k \phi_k^* \psi_k$$
   - **Orthogonality:** If $\langle \phi | \psi \rangle = 0$, the states are mutually orthogonal and distinguishable with $100\%$ certainty in projective measurement.
   - **Overlap Amplitude:** For non-orthogonal states, $|\langle \phi | \psi \rangle|^2$ gives the probability of observing state $|\phi\rangle$ when measuring $|\psi\rangle$.
2. **Outer Product (Ket-Bra $|\psi\rangle \langle \phi |$):**
   Produces a rank-1 operator in $\mathbb{C}^{2 \times 2}$:
   $$|\psi\rangle \langle \phi | = \psi \phi^\dagger = \begin{pmatrix} \psi_0 \phi_0^* & \psi_0 \phi_1^* \\ \psi_1 \phi_0^* & \psi_1 \phi_1^* \end{pmatrix}$$
   - **Subspace Projectors:** $P_0 = |0\rangle \langle 0|$ and $P_1 = |1\rangle \langle 1|$ project onto orthogonal computational axes.
   - **Completeness Relation (Resolution of the Identity):**
     $$P_0 + P_1 = |0\rangle \langle 0| + |1\rangle \langle 1| = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix} + \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = I_2$$

**Prediction Check:**
- $\langle 0 | 1 \rangle = (1)(0) + (0)(1) = 0.0$ (perfect orthogonality).
- $\langle 0 | + \rangle = (1)(1/\sqrt{2}) + (0)(1/\sqrt{2}) = 1/\sqrt{2} \approx 0.707107$.
- Projector sum: $P_0 + P_1 = I_2$.

In [2]:
# Experiment 2: Inner products, orthogonality, and completeness relation
print("=== Inner & Outer Products ===")

# Inner product <0|1>
inner_0_1 = ket0.conj().T @ ket1
print(f"<0|1> = {inner_0_1.real:+.4f} (Expected: 0.0, orthogonal)")
assert np.isclose(inner_0_1, 0.0), "<0|1> must equal 0"

# Inner product <0|+>
inner_0_plus = ket0.conj().T @ ket_plus
expected_overlap = 1.0 / np.sqrt(2)
print(f"<0|+> = {inner_0_plus.real:+.4f} (Expected: 1/sqrt(2) ≈ 0.7071)")
assert np.isclose(inner_0_plus, expected_overlap), "<0|+> must equal 1/sqrt(2)"

# Outer products (Projectors)
P0 = np.outer(ket0, ket0.conj())  # |0><0|
P1 = np.outer(ket1, ket1.conj())  # |1><1|

print("\nProjector P0 = |0><0|:\n", P0.real)
print("Projector P1 = |1><1|:\n", P1.real)

# Completeness relation: P0 + P1 = I_2
I2 = np.eye(2, dtype=complex)
completeness_sum = P0 + P1
print("\nP0 + P1:\n", completeness_sum.real)
assert np.allclose(completeness_sum, I2), "P0 + P1 must equal the 2x2 Identity!"
print("\nCompleteness Check: PASS (P0 + P1 == I2)")

=== Inner & Outer Products ===
<0|1> = +0.0000 (Expected: 0.0, orthogonal)
<0|+> = +0.7071 (Expected: 1/sqrt(2) ≈ 0.7071)

Projector P0 = |0><0|:
 [[1. 0.]
 [0. 0.]]
Projector P1 = |1><1|:
 [[0. 0.]
 [0. 1.]]

P0 + P1:
 [[1. 0.]
 [0. 1.]]

Completeness Check: PASS (P0 + P1 == I2)


## 3. Unitary & Hermitian Operators

**Mathematical Definitions:**
1. **Unitary Operators ($U^\dagger U = I$):**
   Physical evolution must preserve statevector norm: $\langle U\psi | U\psi \rangle = \langle \psi | U^\dagger U | \psi \rangle = \langle \psi | \psi \rangle = 1$. Hence, every quantum gate matrix must be unitary ($U^\dagger U = I$). All eigenvalues have unit modulus ($|\lambda| = 1$).
2. **Hermitian Operators ($M^\dagger = M$):**
   Physical observables and Hamiltonians must have real expectation values and real eigenvalues (measurable quantities in experiments). Hence, every observable matrix must be Hermitian ($M^\dagger = M$).

**Testing Canonical Quantum Gates:**
- Pauli matrices: $X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$, $Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$, $Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$
- Hadamard: $H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$
- Phase gates: $S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}$, $T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix}$
- Rotation: $R_Y(\pi/3) = \begin{pmatrix} \cos(\pi/6) & -\sin(\pi/6) \\ \sin(\pi/6) & \cos(\pi/6) \end{pmatrix} = \begin{pmatrix} \sqrt{3}/2 & -1/2 \\ 1/2 & \sqrt{3}/2 \end{pmatrix}$

**Prediction Check:**
- $X, Y, Z, H$: Both **Unitary** ($U^\dagger U = I$) AND **Hermitian** ($U^\dagger = U$). These gates can serve both as reversible quantum operations and as measurable observables.
- $S, T, R_Y(\pi/3)$: **Unitary** ($U^\dagger U = I$) but **NOT Hermitian** ($U^\dagger \neq U$). These can serve as quantum state preparation gates, but not as physical observables directly.

In [3]:
# Experiment 3: Unitary and Hermitian checks across standard quantum gates
def is_unitary(U):
    """Checks if U^dagger @ U == I."""
    d = U.shape[0]
    return np.allclose(U.conj().T @ U, np.eye(d, dtype=complex))

def is_hermitian(M):
    """Checks if M^dagger == M."""
    return np.allclose(M.conj().T, M)

# Define gate matrices
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
S = np.array([[1, 0], [0, 1j]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)
Ry_pi3 = np.array([[np.cos(np.pi/6), -np.sin(np.pi/6)],
                   [np.sin(np.pi/6),  np.cos(np.pi/6)]], dtype=complex)

gates = {
    "X": X, "Y": Y, "Z": Z, "H": H,
    "S": S, "T": T, "Ry(pi/3)": Ry_pi3
}

print(f"{'Gate':<10} | {'Unitary?':<10} | {'Hermitian?':<12} | Status")
print("-" * 52)
for name, mat in gates.items():
    u = is_unitary(mat)
    h = is_hermitian(mat)
    status = "Both (Gate & Observable)" if (u and h) else "Unitary only (Gate only)"
    print(f"{name:<10} | {str(u):<10} | {str(h):<12} | {status}")
    
    # Strict assertion checks
    assert u, f"Gate {name} must be unitary!"
    if name in ["X", "Y", "Z", "H"]:
        assert h, f"Gate {name} must be Hermitian!"
    else:
        assert not h, f"Gate {name} must NOT be Hermitian!"

print("\nUnitary & Hermitian Check: ALL PASS")

Gate       | Unitary?   | Hermitian?   | Status
----------------------------------------------------
X          | True       | True         | Both (Gate & Observable)
Y          | True       | True         | Both (Gate & Observable)
Z          | True       | True         | Both (Gate & Observable)
H          | True       | True         | Both (Gate & Observable)
S          | True       | False        | Unitary only (Gate only)
T          | True       | False        | Unitary only (Gate only)
Ry(pi/3)   | True       | False        | Unitary only (Gate only)

Unitary & Hermitian Check: ALL PASS


## 4. Kronecker (Tensor) Products: Multi-Qubit State & Operator Construction

**Mathematical Definition:**
When composite quantum systems are constructed, their joint Hilbert space is the tensor product $\mathcal{H}_1 \otimes \mathcal{H}_2$.
1. **Basis States:**
   The 4-dimensional basis states for a 2-qubit system are formed by the Kronecker product of single-qubit basis states:
   $$|00\rangle = |0\rangle \otimes |0\rangle = \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix}, \quad |01\rangle = |0\rangle \otimes |1\rangle = \begin{pmatrix} 0 \\ 1 \\ 0 \\ 0 \end{pmatrix}$$
   $$|10\rangle = |1\rangle \otimes |0\rangle = \begin{pmatrix} 0 \\ 0 \\ 1 \\ 0 \end{pmatrix}, \quad |11\rangle = |1\rangle \otimes |1\rangle = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}$$
2. **Multi-Qubit Operators:**
   - Single-qubit operation on qubit 0 with identity on qubit 1:
     $$H \otimes I = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} \otimes \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \\ 1 & 0 & -1 & 0 \\ 0 & 1 & 0 & -1 \end{pmatrix}$$
   - Two-qubit Pauli Ising interaction:
     $$Z \otimes Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \otimes \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & -1 & 0 & 0 \\ 0 & 0 & -1 & 0 \\ 0 & 0 & 0 & 1 \end{pmatrix}$$

**Prediction Check:**
- Basis states are orthonormal vectors in $\mathbb{C}^4$ with a single non-zero entry `1.0` at indices 0, 1, 2, 3.
- $Z \otimes Z$ is diagonal with eigenvalues $[+1, -1, -1, +1]$:
  - $(Z \otimes Z)|00\rangle = (+1)(+1)|00\rangle = +1 |00\rangle$
  - $(Z \otimes Z)|01\rangle = (+1)(-1)|01\rangle = -1 |01\rangle$
  - $(Z \otimes Z)|10\rangle = (-1)(+1)|10\rangle = -1 |10\rangle$
  - $(Z \otimes Z)|11\rangle = (-1)(-1)|11\rangle = +1 |11\rangle$

In [4]:
# Experiment 4: Kronecker products for basis states and operators
# Construct 2-qubit computational basis states via np.kron
ket00 = np.kron(ket0, ket0)  # Index 0
ket01 = np.kron(ket0, ket1)  # Index 1
ket10 = np.kron(ket1, ket0)  # Index 2
ket11 = np.kron(ket1, ket1)  # Index 3

basis_2q = {
    "|00>": ket00,
    "|01>": ket01,
    "|10>": ket10,
    "|11>": ket11
}

print("=== Two-Qubit Basis Vectors (np.kron) ===")
for name, vec in basis_2q.items():
    print(f"{name}: {vec.real.astype(int)}")
    assert np.linalg.norm(vec) == 1.0

# Construct multi-qubit operators
H_kron_I = np.kron(H, I2)
Z_kron_Z = np.kron(Z, Z)

print("\nOperator H (x) I (shape 4x4):\n", np.round(H_kron_I.real, 4))
print("\nOperator Z (x) Z (shape 4x4):\n", Z_kron_Z.real.astype(int))
print("Diagonal of Z (x) Z:", np.diag(Z_kron_Z).real.astype(int))

# Verification of eigenvalues and unitarity
assert np.allclose(np.diag(Z_kron_Z), [1, -1, -1, 1])
assert is_unitary(H_kron_I) and is_unitary(Z_kron_Z)
print("\nKronecker Operator Check: PASS (Diagonal [1, -1, -1, 1] verified for ZZ)")

=== Two-Qubit Basis Vectors (np.kron) ===
|00>: [1 0 0 0]
|01>: [0 1 0 0]
|10>: [0 0 1 0]
|11>: [0 0 0 1]

Operator H (x) I (shape 4x4):
 [[ 0.7071  0.      0.7071  0.    ]
 [ 0.      0.7071  0.      0.7071]
 [ 0.7071  0.     -0.7071 -0.    ]
 [ 0.      0.7071 -0.     -0.7071]]

Operator Z (x) Z (shape 4x4):
 [[ 1  0  0  0]
 [ 0 -1  0  0]
 [ 0  0 -1  0]
 [ 0  0  0  1]]
Diagonal of Z (x) Z: [ 1 -1 -1  1]

Kronecker Operator Check: PASS (Diagonal [1, -1, -1, 1] verified for ZZ)


## 5. Bridge to QAOA Max-Cut Cost Hamiltonian

**Mathematical Derivation:**
For a graph $G = (V, E)$, the classical Max-Cut problem partitions vertices $V$ into two disjoint sets $S$ and $\bar{S}$ to maximize the number (or weight) of edges crossing the cut.
Let binary variable $x_i \in \{0, 1\}$ indicate whether vertex $i$ belongs to $S$ ($x_i=0$) or $\bar{S}$ ($x_i=1$).
For any edge $(i, j) \in E$, the edge is cut if $x_i \neq x_j$:
$$c_{ij}(x_i, x_j) = x_i(1 - x_j) + x_j(1 - x_i)$$
In physics and quantum optimization, we map classical binary variables $x_i \in \{0, 1\}$ to Ising spin variables $z_i \in \{+1, -1\}$ via:
$$z_i = 1 - 2x_i \iff x_i = \frac{1 - z_i}{2}$$
Substituting into the edge cut equation:
$$c_{ij} = \frac{1 - z_i z_j}{2} = \frac{1}{2}(1 - z_i z_j)$$
Promoting classical spin variables $z_i$ to quantum Pauli $Z$ operators gives the single-edge Max-Cut Hamiltonian:
$$C_{ij} = \frac{1}{2}(I_4 - Z \otimes Z)$$

**Action on Basis States:**
- For $|00\rangle$ ($x_i=0, x_j=0$, same side): $C_{ij}|00\rangle = \frac{1}{2}(1 - (+1)(+1))|00\rangle = 0|00\rangle$
- For $|01\rangle$ ($x_i=0, x_j=1$, cut edge): $C_{ij}|01\rangle = \frac{1}{2}(1 - (+1)(-1))|01\rangle = 1|01\rangle$
- For $|10\rangle$ ($x_i=1, x_j=0$, cut edge): $C_{ij}|10\rangle = \frac{1}{2}(1 - (-1)(+1))|10\rangle = 1|10\rangle$
- For $|11\rangle$ ($x_i=1, x_j=1$, same side): $C_{ij}|11\rangle = \frac{1}{2}(1 - (-1)(-1))|11\rangle = 0|11\rangle$

**Prediction Check:**
- The matrix $C_{ij} = \frac{1}{2}(I_4 - Z \otimes Z)$ is diagonal with entries $[0, 1, 1, 0]$.
- Matrix expectation values $\langle x | C_{ij} | x \rangle = x^\dagger C_{ij} x$ must evaluate to:
  - $0$ for states $|00\rangle$ and $|11\rangle$ (uncut edges).
  - $1$ for states $|01\rangle$ and $|10\rangle$ (cut edges).

In [5]:
# Experiment 5: Single-edge Max-Cut operator C_ij = 0.5 * (I4 - Z (x) Z)
I4 = np.eye(4, dtype=complex)
C_ij = 0.5 * (I4 - Z_kron_Z)

print("=== Single-Edge Max-Cut Hamiltonian C_ij ===")
print("Matrix C_ij:\n", C_ij.real)
print("Diagonal elements of C_ij:", np.diag(C_ij).real)

# Verify diagonal matches exact expected vector [0, 1, 1, 0]
assert np.allclose(np.diag(C_ij).real, [0.0, 1.0, 1.0, 0.0]), "Diagonal must be [0, 1, 1, 0]!"

# Verify expectation value <x|C_ij|x> via matrix multiplication
print("\nExpectation values <x|C_ij|x>:")
for name, state in basis_2q.items():
    # Matrix-vector product: state^dagger @ C_ij @ state
    exp_val = (state.conj().T @ C_ij @ state).real
    partition = "SAME partition (uncut)" if exp_val == 0.0 else "DIFFERENT partition (cut)"
    print(f"<{name}| C_ij |{name}> = {exp_val:.1f}  -->  {partition}")
    
    if name in ["|00>", "|11>"]:
        assert np.isclose(exp_val, 0.0), f"State {name} should have 0 cut value (same partition)!"
    else:
        assert np.isclose(exp_val, 1.0), f"State {name} should have 1 cut value (separated edge)!"

print("\nQAOA Max-Cut Bridge Check: ALL PASS (C_ij correctly scores graph cuts via matrix multiplication)")

=== Single-Edge Max-Cut Hamiltonian C_ij ===
Matrix C_ij:
 [[0. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 0.]]
Diagonal elements of C_ij: [0. 1. 1. 0.]

Expectation values <x|C_ij|x>:
<|00>| C_ij ||00>> = 0.0  -->  SAME partition (uncut)
<|01>| C_ij ||01>> = 1.0  -->  DIFFERENT partition (cut)
<|10>| C_ij ||10>> = 1.0  -->  DIFFERENT partition (cut)
<|11>| C_ij ||11>> = 0.0  -->  SAME partition (uncut)

QAOA Max-Cut Bridge Check: ALL PASS (C_ij correctly scores graph cuts via matrix multiplication)
